In [ ]:
# --- ENVIRONMENT SETUP: Environment-Aware Paths ---
import sys, os
from pathlib import Path

try:
    # Standard Colab setup
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = Path(str(REPO_ROOT))
except ImportError:
    # Local fallback (assumes running from explorations/ or experiments/)
    cur = Path().resolve()
    REPO_ROOT = cur.parent if cur.name in ('explorations', 'experiments') else cur.parents[1]

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config.paths import GOLD, BRONZE, SILVER, DATA_LAKE, MODELS, PRETRAINED, TRAINED, OPS, MLFLOW_TRACKING_URI, REPOS


# {EXPERIMENT_NAME}

**Fire-and-forget** — Run All → Close Tab → check `batch_status.ipynb` later

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%pip install -q -r requirements.txt


In [ ]:
%cd /content/drive/MyDrive/repos/deep-learning/experiments/_template
%run train.py

In [ ]:
import json
from pathlib import Path

marker = Path('completed.json')
if marker.exists():
    data = json.loads(marker.read_text())
    status = '✅' if data['success'] else '❌'
    print(f"{status} {data['experiment']}")
    print(f"   Best acc: {data['best_val_acc']}")
    print(f"   Duration: {data['duration_seconds']/60:.1f}m")
    print(f"   Model:    {data['model']}")
else:
    print('⚠️ No completion marker found')

In [ ]:
import mlflow
import torch
client = mlflow.tracking.MlflowClient()

# Find the best run dynamically
experiment = client.get_experiment_by_name('deep-learning')
if experiment:
    best_run = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=["metrics.best_val_acc DESC"],
        max_results=1
    )[0]
    print(f"Best Run ID: {best_run.info.run_id}")
    print(f"Best Val Acc: {best_run.data.metrics.get('best_val_acc', 'N/A')}")
    
    # Since we saved raw weights using `torch.save()`, we download the artifact
    # weights_path = mlflow.artifacts.download_artifacts(f"runs:/{best_run.info.run_id}/best_model.pt")
    # print(f"Model downloaded to: {weights_path}")
    
    # Now you can load it into your PyTorch model:
    # model = get_model(config).to(device, non_blocking=True)  # Re-initialize your blank network here
    # model.load_state_dict(torch.load(weights_path, map_location=device))
    # print("Weights loaded successfully!")
else:
    print('Experiment not found')


In [ ]:
from google.colab import runtime
runtime.unassign()
